# Exercises: Advanced ML

Gradient boosting, imbalanced learning, custom transformers, and ensembles.

## Exercise 1: XGBoost vs LightGBM Comparison

Train both boosting libraries on the same dataset and compare:
- Accuracy / F1
- Training time
- Feature importance rankings

**Requirements:**
- Synthetic dataset with 1000 samples, 20 features
- Matched hyperparameters where possible
- 5-fold CV

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import lightgbm as lgb

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20,
                            n_informative=12, random_state=42)

shared_params = dict(n_estimators=200, max_depth=5, learning_rate=0.1,
                      random_state=42)

xgb_model = xgb.XGBClassifier(**shared_params, use_label_encoder=False,
                                eval_metric='logloss', verbosity=0)
lgb_model = lgb.LGBMClassifier(**shared_params, verbose=-1)

results = {}
for name, model in [('XGBoost', xgb_model), ('LightGBM', lgb_model)]:
    t0 = time.time()
    acc = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    f1 = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
    elapsed = time.time() - t0
    results[name] = {'Accuracy': f'{acc.mean():.4f}',
                     'F1': f'{f1.mean():.4f}',
                     'Time (s)': f'{elapsed:.2f}'}

print(pd.DataFrame(results).T.to_string())

# Feature importance comparison
xgb_model.fit(X, y); lgb_model.fit(X, y)
fi = pd.DataFrame({
    'XGB_importance': xgb_model.feature_importances_,
    'LGB_importance': lgb_model.feature_importances_,
}, index=[f'f{i}' for i in range(20)])
fi['XGB_rank'] = fi['XGB_importance'].rank(ascending=False).astype(int)
fi['LGB_rank'] = fi['LGB_importance'].rank(ascending=False).astype(int)
print("\nTop 5 features by XGBoost:")
print(fi.sort_values('XGB_rank').head(5)[['XGB_rank', 'LGB_rank']])


### Explanation

Both libraries implement gradient boosting but differ in tree-building: XGBoost uses level-wise growth, LightGBM uses leaf-wise (faster, sometimes more accurate). Matched hyper-parameters make the comparison fair. Feature importance rankings often agree on top features but diverge on less important ones.

## Exercise 2: SMOTE Analysis

Compare model performance **with and without SMOTE** on an imbalanced dataset (95/5 split).

**Requirements:**
- Create a dataset with `weights=[0.95, 0.05]`
- Train `RandomForestClassifier` without SMOTE
- Train with SMOTE using `imblearn.pipeline.Pipeline`
- Compare precision, recall, and F1 for the minority class

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=15,
                            n_informative=10, weights=[0.95, 0.05],
                            flip_y=0, random_state=42)
print(f"Class distribution: {np.bincount(y)}")

scorers = {
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
}

# Without SMOTE
rf = RandomForestClassifier(n_estimators=100, random_state=42)
no_smote = cross_validate(rf, X, y, cv=5, scoring=scorers)

# With SMOTE
smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
])
with_smote = cross_validate(smote_pipe, X, y, cv=5, scoring=scorers)

print(f"\n{'Metric':<12} {'No SMOTE':>10} {'With SMOTE':>12}")
print("-" * 36)
for metric in ['precision', 'recall', 'f1']:
    key = f'test_{metric}'
    a = no_smote[key].mean()
    b = with_smote[key].mean()
    print(f"{metric:<12} {a:>10.4f} {b:>12.4f}")


### Explanation

SMOTE synthesises minority-class samples by interpolating between nearest neighbours. Using `imblearn.pipeline.Pipeline` ensures SMOTE is applied only to training folds, preventing data leakage. SMOTE typically boosts recall at the cost of some precision.

## Exercise 3: Custom Sklearn Transformer

Create a custom transformer that:
1. Computes the **log1p** of specified columns
2. Adds **interaction features** (products of column pairs)
3. Works inside a sklearn `Pipeline`

**Requirements:**
- Subclass `BaseEstimator` + `TransformerMixin`
- Implement `fit` and `transform`
- Verify with `check_estimator` (optional) and a pipeline test

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification
from itertools import combinations


class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Adds log1p and pairwise interaction features."""
    def __init__(self, log_cols=None, interaction_cols=None):
        self.log_cols = log_cols        # list of column indices
        self.interaction_cols = interaction_cols  # list of column indices

    def fit(self, X, y=None):
        return self  # stateless transformer

    def transform(self, X):
        X = np.array(X, dtype=float)
        parts = [X]
        if self.log_cols is not None:
            log_feats = np.log1p(np.abs(X[:, self.log_cols]))
            parts.append(log_feats)
        if self.interaction_cols is not None:
            for i, j in combinations(self.interaction_cols, 2):
                parts.append((X[:, i] * X[:, j]).reshape(-1, 1))
        return np.hstack(parts)


np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=6,
                            n_informative=4, random_state=42)

# Baseline
base_score = cross_val_score(LogisticRegression(max_iter=300),
                              X, y, cv=5).mean()

# With custom transformer
pipe = Pipeline([
    ('feat_eng', FeatureEngineer(log_cols=[0, 1, 2],
                                  interaction_cols=[0, 1, 2, 3])),
    ('clf', LogisticRegression(max_iter=300)),
])
eng_score = cross_val_score(pipe, X, y, cv=5).mean()

print(f"Baseline accuracy:   {base_score:.4f}")
print(f"Engineered accuracy: {eng_score:.4f}")
print(f"New features added:  log1p(3) + C(4,2)=6 interactions = 9")


### Explanation

By subclassing `BaseEstimator` and `TransformerMixin`, the transformer gets `get_params`/`set_params` (needed for `GridSearchCV`) and a default `fit_transform` for free. A stateless transformer just returns `self` from `fit`.

## Exercise 4: Ensemble Stacking

Build a **stacking ensemble** with:
- Base learners: Ridge, RandomForest, SVM
- Meta-learner: LogisticRegression
- Use `StackingClassifier` from sklearn
- Compare stacking accuracy vs each base learner alone

**Requirements:**
- 5-fold CV for evaluation
- Print a comparison table

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC

np.random.seed(42)
X, y = make_classification(n_samples=800, n_features=20,
                            n_informative=12, random_state=42)

base_learners = [
    ('ridge', RidgeClassifier()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(kernel='rbf', random_state=42)),
]

stack = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=500),
    cv=5,
)

results = {}
for name, model in base_learners:
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name:>10}: {scores.mean():.4f} ± {scores.std():.4f}")

stack_scores = cross_val_score(stack, X, y, cv=5, scoring='accuracy')
results['Stacking'] = stack_scores.mean()
print(f"{'Stacking':>10}: {stack_scores.mean():.4f} ± {stack_scores.std():.4f}")

best = max(results, key=results.get)
print(f"\nBest model: {best}")


### Explanation

Stacking trains base learners on CV folds, collects their out-of-fold predictions as new features, and feeds them to a meta-learner. This captures complementary strengths of diverse models. The meta-learner learns which base learner to trust in which region of feature space.